# LoRA and QLoRA Mechanics

Full fine-tuning updates all model weights. [Parameter-efficient fine-tuning (PEFT)](https://huggingface.co/docs/peft/index) updates a much smaller set while the pretrained base model stays frozen. This notebook makes that difference measurable before training.

You will inspect trainable parameters, connect LoRA hyperparameters to the low-rank update, and choose between full fine-tuning, LoRA, and QLoRA for different hardware constraints.


```mermaid
flowchart LR
    A["Pretrained base model"] --> B["Freeze original weights"]
    B --> C["Insert trainable adapters"]
    C --> D["Train adapters on examples"]
    D --> E["Load base + adapter at inference"]
```

[LoRA](https://arxiv.org/abs/2106.09685) represents a weight update as two small matrices, commonly written as $\Delta W = BA$. If the original layer has shape $d_{out} \times d_{in}$ and the adapter rank is $r$, LoRA learns roughly $r(d_{in} + d_{out})$ values instead of $d_{in}d_{out}$ values. [QLoRA](https://arxiv.org/abs/2305.14314) adds quantized storage for the frozen base model while the LoRA adapters remain trainable.


**LoRA adapter mechanism.** Source: [Hugging Face PEFT LoRA diagram](https://huggingface.co/datasets/huggingface/documentation-images/blob/main/peft/lora_diagram.png). The frozen base layer is combined with a small trainable low-rank update.

<img src="assets/hf-lora-diagram.png" alt="LoRA adapter diagram" style="max-width: 700px; width: 100%; height: auto;">


In [ ]:
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
DEVICE = torch.device("cpu")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

## Count Parameters

Counting trainable parameters turns adapter training from an abstract concept into something you can inspect. The same check is useful in real projects before launching expensive training jobs.

Parameter count is only one part of memory use. Training also stores gradients, activations, and optimizer state. LoRA reduces gradient and optimizer-state memory because most base parameters stay frozen, but activations still depend on sequence length, batch size, and model architecture.


In [ ]:
def parameter_summary(model_to_count: torch.nn.Module) -> dict[str, float]:
    # Count all tensor elements, then count only tensors that require gradients.
    total = sum(parameter.numel() for parameter in model_to_count.parameters())
    trainable = sum(
        parameter.numel()
        for parameter in model_to_count.parameters()
        if parameter.requires_grad
    )
    return {
        "trainable": trainable,
        "total": total,
        "trainable_percent": round(100 * trainable / total, 4),
    }


parameter_summary(model)

### Exercise: Interpret Full Fine-Tuning Cost

Explain why full fine-tuning becomes expensive for large models.


### Solution

Full fine-tuning is expensive because gradients and optimizer state must be stored for all model parameters, so memory and compute scale with the full model size.


## Add a LoRA Adapter

SmolLM2 uses separate query and value projections inside each attention block. We target `q_proj` and `v_proj`: the frozen projections still perform the original computation, while LoRA learns small task-specific updates.

The main adapter choices are:

- `r`: rank, which controls adapter capacity and parameter count;
- `lora_alpha`: scaling applied to the learned update;
- `lora_dropout`: regularization applied on the adapter path during training;
- `target_modules`: the layer types that receive adapters.


In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)

lora_model = get_peft_model(model, lora_config)
parameter_summary(lora_model)

### Exercise: Inspect Adapter Targets

List the first four module names ending in `q_proj` or `v_proj`.


In [ ]:
# TODO: inspect named modules and keep query/value projection names.


In [ ]:
# Solution
adapter_target_names = [
    name
    for name, _ in lora_model.named_modules()
    if name.endswith(("q_proj", "v_proj"))
]
adapter_target_names[:4]

### Exercise: Compute Frozen Parameters

Use the summary dictionary to compute how many parameters are frozen after adding LoRA.


In [ ]:
# TODO: compute frozen parameter count from parameter_summary(lora_model).


In [ ]:
# Solution
lora_counts = parameter_summary(lora_model)
frozen_parameters = lora_counts["total"] - lora_counts["trainable"]
frozen_parameters

**Prompt tuning as learned soft context.** Source: [Hugging Face documentation images](https://huggingface.co/datasets/huggingface/documentation-images/blob/main/peft/prompt-tuning.png). Prompt tuning learns trainable prompt embeddings rather than ordinary text tokens. Unlike LoRA, it changes the learned input context rather than inserting low-rank updates into attention projections.

<img src="assets/hf-prompt-tuning.png" alt="Prompt tuning diagram" style="max-width: 660px; width: 100%; height: auto;">


Prompt tuning is attractive when a very small task-specific parameter set is enough. Prefix tuning reaches deeper into the model by adding learned vectors to attention-related hidden states, so the location of adaptation differs even though both methods keep the base model frozen.


**Prefix tuning mechanism.** Source: [Hugging Face documentation images](https://huggingface.co/datasets/huggingface/documentation-images/blob/main/peft/prefix-tuning.png). Prefix tuning adds learned vectors that influence hidden states, which is different from writing a longer prompt.

<img src="assets/hf-prefix-tuning.png" alt="Prefix tuning diagram" style="max-width: 660px; width: 100%; height: auto;">


## Where QLoRA Fits

QLoRA combines quantization and LoRA. The base model is stored in low precision while adapter weights remain trainable, reducing memory use while adding quantization-specific complexity.

| Approach | Base weights | Updated parameters | Typical reason to choose it |
| --- | --- | --- | --- |
| Full fine-tuning | Full precision | All weights | Maximum flexibility when compute and data are abundant. |
| LoRA | Usually full or half precision | Low-rank adapters | Lower training cost and small portable checkpoints. |
| QLoRA | Usually 4-bit quantized | Low-rank adapters | Fit a larger base model into limited accelerator memory. |

Quantization is not automatically better. It can introduce hardware constraints, numerical trade-offs, and extra configuration. For this CPU-friendly repository, ordinary LoRA keeps the runnable path simpler; QLoRA is taught as the next scaling step.


**Quantization visual.** Source: [Hugging Face documentation images](https://huggingface.co/datasets/huggingface/documentation-images/blob/main/transformers/quant_visual.png). Quantization stores weights with fewer bits, reducing memory pressure at the cost of precision trade-offs.

<img src="assets/hf-quant-visual.png" alt="Quantization visual" style="max-width: 700px; width: 100%; height: auto;">


**Dequantization visual.** Source: [Hugging Face documentation images](https://huggingface.co/datasets/huggingface/documentation-images/blob/main/transformers/dequant.png). This separates compact storage from compute-time representation, which is central to understanding QLoRA.

<img src="assets/hf-dequant.png" alt="Dequantization visual" style="max-width: 700px; width: 100%; height: auto;">


### Exercise: Choose LoRA or QLoRA

For a 7B parameter model on a limited-memory GPU, which approach is more likely to fit: full fine-tuning, LoRA, or QLoRA?


### Solution

QLoRA is usually most memory-efficient because the base model can be loaded in low precision while only adapter parameters are trained. LoRA also reduces memory, but typically less than QLoRA.


**P-tuning as another PEFT family member.** Source: [Hugging Face documentation images](https://huggingface.co/datasets/huggingface/documentation-images/blob/main/peft/p-tuning.png). The broader PEFT family includes several ways to adapt behavior without full fine-tuning.

<img src="assets/hf-p-tuning.png" alt="P-tuning diagram" style="max-width: 700px; width: 100%; height: auto;">


## Further Reading

- [Hugging Face PEFT LoRA guide](https://huggingface.co/docs/peft/developer_guides/lora)
- [LoRA paper](https://arxiv.org/abs/2106.09685)
- [QLoRA paper](https://arxiv.org/abs/2305.14314)
- [Hugging Face quantization overview](https://huggingface.co/docs/transformers/quantization/overview)
